# ConvNeXt-Temporal Transformer Frame Generator (Y Channel)

Generative Replacement of Predictive Frame (G) - Luminance (Y) Only

**Architecture:**
1. **Shared ConvNeXt Encoder** - 4 stages (96/192/384/768 ch), shared across all T frames
2. **Temporal Transformer** - spatio-temporal fusion over T*256 tokens
3. **ConvNeXt Decoder** - 5x Up-x2 blocks + 3x3 Conv to 1-channel Y frame

**Dataset:** Inter4K RAW `.npy`, input `(7, 512, 512)`, output `(1, 512, 512)`.  
Files are loaded **sample-by-sample** (no full dataset iteration).

In [ ]:
import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

ON_KAGGLE = os.path.exists('/kaggle/working')

if ON_KAGGLE:
    # Kaggle: dataset attached as input, outputs go to /kaggle/working
    DATASET_ROOT = '/kaggle/input/datasets/tonmoyk983/sevtone-4-qp-gop8/sevtone_4_QP_GOP8/Inter4K/RAW'
    CKPT_DIR     = '/kaggle/working'
else:
    # Local Windows
    DATASET_ROOT = r'D:\Dataset\Inter4K\60fps\UHD\Segments\sevtone_4_QP_GOP8\Inter4K\RAW'
    CKPT_DIR     = r'D:\Dataset\Inter4K\60fps\UHD\Segments\sevtone_4_QP_GOP8'

CKPT_PATH = os.path.join(CKPT_DIR, 'best_model.pth')
os.makedirs(CKPT_DIR, exist_ok=True)

print(f'Running on: {"Kaggle" if ON_KAGGLE else "Local"}')
print(f'DATASET_ROOT : {DATASET_ROOT}')
print(f'CKPT_PATH    : {CKPT_PATH}')

# ── Model / training config ───────────────────────────────────────────────────
T             = 7          # input frames per sample (GOP-8)
IMG_SIZE      = 512        # spatial size fed to the model
# PATCH_SIZE is auto-derived: stem /4, then three /2 stages = /32 total
# e.g. IMG_SIZE=512 -> 512/32=16  |  IMG_SIZE=224 -> 224/32=7
PATCH_SIZE    = IMG_SIZE // 32   # (B, 768, PATCH_SIZE, PATCH_SIZE) after encoder
assert IMG_SIZE % 32 == 0, f'IMG_SIZE must be divisible by 32, got {IMG_SIZE}'
EMBED_DIM     = 768        # channels at Stage 4
NUM_HEADS     = 8          # multi-head attention heads
TRANS_LAYERS  = 4          # transformer encoder layers L
MLP_RATIO     = 4          # FFN expansion ratio
BATCH_SIZE    = 2          # reduce if OOM; increase if VRAM allows
USE_AMP       = True       # mixed precision (float16 forward) - cuts VRAM ~50%
LEARNING_RATE = 1e-4
EPOCHS        = 30
# ── Resume config ─────────────────────────────────────────────────────────────
# On Kaggle: attach your previous notebook output as an Input, then set:
#   RESUME_CKPT_PATH = '/kaggle/input/<your-notebook-slug>/best_model.pth'
# Leave as None to train from scratch.
RESUME_CKPT_PATH = None
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device        : {DEVICE}")
print(f"T={T} | IMG_SIZE={IMG_SIZE} | EMBED_DIM={EMBED_DIM} | TRANS_LAYERS={TRANS_LAYERS}")


## 2. Dataset

The `Inter4KDataset` loads `.npy` files on demand and returns them **as-is** at 512x512.
No resizing or interpolation is applied.

- Input  `sample_input_<id>.npy`  : shape `(7, 512, 512)`, uint8 [0, 255] -> float32 [0, 1]
- Output `sample_output_<id>.npy` : shape `(1, 512, 512)`, uint8 [0, 255] -> float32 [0, 1]

In [ ]:
class Inter4KDataset(Dataset):
    """
    Inter4K RAW dataset.
    Loads .npy files on demand. No resizing - data is used as-is at 512x512.

    Parameters
    ----------
    root       : RAW directory (contains Input/ and Output/ sub-dirs)
    sample_ids : list of integer sample IDs (1-indexed)
    """

    def __init__(self, root: str, sample_ids: list):
        self.input_dir  = os.path.join(root, 'Input')
        self.output_dir = os.path.join(root, 'Output')
        self.ids        = sample_ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        sid     = self.ids[idx]
        inp_arr = np.load(os.path.join(self.input_dir,  f'sample_input_{sid}.npy'))   # (7, 512, 512) uint8
        out_arr = np.load(os.path.join(self.output_dir, f'sample_output_{sid}.npy'))  # (1, 512, 512) uint8
        if out_arr.ndim == 2:
            out_arr = out_arr[np.newaxis]                                              # ensure (1, H, W)
        # uint8 [0,255] -> float32 [0,1], no resize
        x = torch.from_numpy(inp_arr.astype(np.float32) / 255.0)   # (7, 512, 512)
        x = x.unsqueeze(1)                                          # (7, 1, 512, 512) - add channel dim
        y = torch.from_numpy(out_arr.astype(np.float32) / 255.0)   # (1, 512, 512)
        return x, y


# ── Build train / val split ────────────────────────────────────────────────────
SUBSET_SIZE  = 20000    # number of samples to use; change as needed
VAL_FRACTION = 0.1

all_ids   = list(range(1, SUBSET_SIZE + 1))
split_idx = int(len(all_ids) * (1 - VAL_FRACTION))
train_ids = all_ids[:split_idx]
val_ids   = all_ids[split_idx:]

train_ds = Inter4KDataset(DATASET_ROOT, train_ids)
val_ds   = Inter4KDataset(DATASET_ROOT, val_ids)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} samples | Val: {len(val_ds)} samples')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
print(f'Sample x shape: {train_ds[0][0].shape}  y shape: {train_ds[0][1].shape}')


## 3. Model Architecture

### 3a. ConvNeXt Block

As shown in the architecture diagram:
```
Input (B, C, H, W)
  -> 7x7 Depthwise Conv (C)
  -> LayerNorm (C)
  -> 1x1 Conv C->4C  [Expand]
  -> GELU
  -> 1x1 Conv 4C->C  [Project]
  + Residual
Output (B, C, H, W)
```

In [ ]:
class ChannelLayerNorm(nn.Module):
    """LayerNorm for (B, C, H, W) tensors - normalises over the channel dim."""
    def __init__(self, num_ch, eps=1e-6):
        super().__init__()
        self.ln = nn.LayerNorm(num_ch, eps=eps)

    def forward(self, x):
        # x: (B, C, H, W) -> permute -> LN -> permute back
        return self.ln(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)


class ConvNeXtBlock(nn.Module):
    """
    Single ConvNeXt block (from architecture diagram):
    7x7 DW-Conv -> LayerNorm -> Linear(C->4C) -> GELU -> Linear(4C->C) -> Residual
    """
    def __init__(self, dim):
        super().__init__()
        self.dw_conv    = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.layer_norm = nn.LayerNorm(dim, eps=1e-6)
        self.pw_expand  = nn.Linear(dim, 4 * dim)
        self.act        = nn.GELU()
        self.pw_project = nn.Linear(4 * dim, dim)

    def forward(self, x):
        residual = x
        x = self.dw_conv(x)               # (B, C, H, W)
        x = x.permute(0, 2, 3, 1)        # (B, H, W, C)
        x = self.layer_norm(x)
        x = self.pw_expand(x)             # (B, H, W, 4C)
        x = self.act(x)
        x = self.pw_project(x)            # (B, H, W, C)
        return x.permute(0, 3, 1, 2) + residual  # (B, C, H, W)


class ConvNeXtStage(nn.Module):
    """
    One encoder stage.
    - is_stem=True  : 4x4 stride-4 conv (Stage 1, 512->128)
    - is_stem=False : LayerNorm + 2x2 stride-2 conv (Stages 2-4, PatchMerge)
    Followed by num_blocks ConvNeXt blocks.
    """
    def __init__(self, in_ch, out_ch, num_blocks, is_stem=False):
        super().__init__()
        layers = []
        if is_stem:
            layers += [
                nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=4),
                ChannelLayerNorm(out_ch),
            ]
        else:
            layers += [
                ChannelLayerNorm(in_ch),
                nn.Conv2d(in_ch, out_ch, kernel_size=2, stride=2),
            ]
        layers += [ConvNeXtBlock(out_ch) for _ in range(num_blocks)]
        self.stage = nn.Sequential(*layers)

    def forward(self, x):
        return self.stage(x)


print("ConvNeXt building blocks defined: ChannelLayerNorm, ConvNeXtBlock, ConvNeXtStage")


### 3b. Shared ConvNeXt Encoder

Applied to each of the T input frames **independently** with **shared weights**.

| Stage | Input Spatial | Output Spatial | Channels | Blocks |
|-------|--------------|----------------|----------|--------|
| 1 (Stem) | 512x512   | 128x128        | 96       | 3      |
| 2     | 128x128      | 64x64          | 192      | 3      |
| 3     | 64x64        | 32x32          | 384      | 9      |
| 4     | 32x32        | 16x16          | 768      | 3      |

In [ ]:
class SharedConvNeXtEncoder(nn.Module):
    """
    Shared ConvNeXt encoder - applied independently to each input Y frame.
    Input : (B, 1, H, W)      single Y-channel frame
    Output: (B, 768, 16, 16)  Stage-4 feature map
    """
    def __init__(self, in_ch=1):
        super().__init__()
        self.stage1 = ConvNeXtStage(in_ch, 96,  num_blocks=3, is_stem=True)   # 512->128
        self.stage2 = ConvNeXtStage(96,  192, num_blocks=3, is_stem=False)  # 128->64
        self.stage3 = ConvNeXtStage(192, 384, num_blocks=9, is_stem=False)  # 64->32
        self.stage4 = ConvNeXtStage(384, 768, num_blocks=3, is_stem=False)  # 32->16

    def forward(self, x):
        x = self.stage1(x)   # (B,  96, 128, 128)
        x = self.stage2(x)   # (B, 192,  64,  64)
        x = self.stage3(x)   # (B, 384,  32,  32)
        x = self.stage4(x)   # (B, 768,  16,  16)
        return x


print("SharedConvNeXtEncoder defined.")


### 3c. Temporal Transformer (Spatio-Temporal Fusion)

```
Per-frame features F0 ... F_{T-1}: each (B, 768, 16, 16)
  -> Flatten spatial: 16x16 = 256 tokens/frame  -> (B, 256, 768)
  -> Concatenate along time                     -> (B, T*256, 768)
  -> Add Temporal Positional Encoding (learnable)
  -> Transformer Encoder (L layers: MHSA + FFN + LayerNorm, Pre-LN)
  -> Take last 256 tokens (target frame slot t=T)
  -> Reshape to spatial feature map             -> (B, 768, 16, 16)
```

In [ ]:
class TemporalTransformer(nn.Module):
    """
    Spatio-Temporal Fusion Transformer.
    Input  : list of T tensors, each (B, 768, 16, 16)
    Output : (B, 768, 16, 16) - fused feature for the target frame
    """
    def __init__(self, embed_dim=768, num_heads=8, num_layers=4,
                 mlp_ratio=4, patch_size=7, T=7, dropout=0.1):
        super().__init__()
        self.T           = T
        self.num_spatial = patch_size * patch_size   # 256 tokens per frame (16x16)
        self.patch_size  = patch_size
        num_tokens       = T * self.num_spatial      # T x 256

        # Learnable temporal positional encoding
        self.pos_embed = nn.Parameter(torch.zeros(1, num_tokens, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * mlp_ratio,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,   # Pre-LayerNorm - more stable
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

    def forward(self, frame_features):
        """
        frame_features : list of T tensors each (B, embed_dim, patch_size, patch_size)
        returns        : (B, embed_dim, patch_size, patch_size)
        """
        B = frame_features[0].shape[0]

        # Flatten spatial -> (B, 256, 768) for each frame
        tokens = [f.flatten(2).transpose(1, 2) for f in frame_features]

        # Concatenate along time -> (B, T*256, 768)
        tokens = torch.cat(tokens, dim=1)

        # Add positional encoding
        tokens = tokens + self.pos_embed

        # Transformer encoder
        fused = self.transformer(tokens)              # (B, T*256, 768)

        # Take last 256 tokens -> target frame slot
        target = fused[:, -self.num_spatial:, :]      # (B, 256, 768)

        # Reshape to spatial feature map
        p = self.patch_size
        return target.transpose(1, 2).view(B, -1, p, p)  # (B, 768, 16, 16)


print("TemporalTransformer defined.")


### 3d. ConvNeXt Decoder (Upsampling)

Five Up-x2 blocks followed by a 3x3 Conv head:

```
16x16,   768 ch
 Up x2 -> 32x32,   384 ch
 Up x2 -> 64x64,   192 ch
 Up x2 -> 128x128,  96 ch
 Up x2 -> 256x256,  64 ch
 Up x2 -> 512x512,  32 ch
 3x3 Conv -> 1 ch (Y)  -> Sigmoid
```

In [ ]:
class UpsampleBlock(nn.Module):
    """Bilinear Up-x2 + 3x3 Conv + ChannelLayerNorm + GELU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm = ChannelLayerNorm(out_ch)
        self.act  = nn.GELU()

    def forward(self, x):
        return self.act(self.norm(self.conv(self.up(x))))


class ConvNeXtDecoder(nn.Module):
    """
    Input : (B, 768, 16, 16)
    Output: (B, 1, 512, 512)  generated Y frame in [0, 1]
    """
    def __init__(self):
        super().__init__()
        self.up1  = UpsampleBlock(768, 384)   # 16  -> 32
        self.up2  = UpsampleBlock(384, 192)   # 32  -> 64
        self.up3  = UpsampleBlock(192,  96)   # 64  -> 128
        self.up4  = UpsampleBlock( 96,  64)   # 128 -> 256
        self.up5  = UpsampleBlock( 64,  32)   # 256 -> 512
        self.head = nn.Conv2d(32, 1, kernel_size=3, padding=1)

    def forward(self, x):
        x = self.up1(x)
        x = self.up2(x)
        x = self.up3(x)
        x = self.up4(x)
        x = self.up5(x)
        return torch.sigmoid(self.head(x))


print("UpsampleBlock, ConvNeXtDecoder defined.")


### 3e. Full Model: ConvNeXtTemporalFrameGenerator

In [ ]:
class ConvNeXtTemporalFrameGenerator(nn.Module):
    """
    ConvNeXt-Temporal Transformer Frame Generator (Y Channel).

    Input : (B, T, 1, H, W)  - T reconstructed Y-channel frames
    Output: (B, 1, H, W)     - generated target Y frame G in [0, 1]

    Pipeline:
    1. Shared ConvNeXt Encoder  (applied independently to each frame)
    2. Temporal Transformer     (spatio-temporal fusion)
    3. ConvNeXt Decoder         (5x upsample to full resolution)
    """
    def __init__(self, T=7, embed_dim=768, num_heads=8,
                 num_layers=4, mlp_ratio=4, patch_size=7, dropout=0.1):
        super().__init__()
        self.T           = T
        self.encoder     = SharedConvNeXtEncoder(in_ch=1)
        self.transformer = TemporalTransformer(
            embed_dim=embed_dim, num_heads=num_heads, num_layers=num_layers,
            mlp_ratio=mlp_ratio, patch_size=patch_size, T=T, dropout=dropout)
        self.decoder     = ConvNeXtDecoder()

    def forward(self, x):
        """
        x : (B, T, 1, H, W)
        """
        B, T_in, C, H, W = x.shape
        assert T_in == self.T, f"Expected T={self.T} input frames, got {T_in}"

        # Stage 1: shared encoder applied independently to each frame
        feats = [self.encoder(x[:, t]) for t in range(T_in)]  # list of (B, 768, 16, 16)

        # Stage 2: temporal transformer
        fused = self.transformer(feats)   # (B, 768, 16, 16)

        # Stage 3: decoder
        return self.decoder(fused)        # (B, 1, H, W)


# ── Instantiate ────────────────────────────────────────────────────────────────
model = ConvNeXtTemporalFrameGenerator(
    T=T, embed_dim=EMBED_DIM, num_heads=NUM_HEADS,
    num_layers=TRANS_LAYERS, mlp_ratio=MLP_RATIO, patch_size=PATCH_SIZE
).to(DEVICE)

total  = sum(p.numel() for p in model.parameters())
trainp = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {total:,} total | {trainp:,} trainable")
print(model)


## 4. Shape Verification

In [ ]:
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, T, 1, IMG_SIZE, IMG_SIZE, device=DEVICE)
    out   = model(dummy)
    print(f"Input  shape : {tuple(dummy.shape)}")
    print(f"Output shape : {tuple(out.shape)}")
    assert out.shape == (2, 1, IMG_SIZE, IMG_SIZE), "Output shape mismatch!"
    print("Shape verification passed")


## 5. Loss, Optimizer & Scheduler

In [ ]:
class CombinedLoss(nn.Module):
    """
    L = alpha * MSE  +  (1 - alpha) * (1 - SSIM)
    Default alpha=0.84 following MS-SSIM convention.
    """
    def __init__(self, alpha=0.84):
        super().__init__()
        self.alpha = alpha
        self.mse   = nn.MSELoss()

    def _gaussian_kernel(self, size, sigma, device):
        coords = torch.arange(size, dtype=torch.float32, device=device) - size // 2
        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
        g /= g.sum()
        k = g.unsqueeze(0) * g.unsqueeze(1)   # (size, size)
        return k.unsqueeze(0).unsqueeze(0)     # (1, 1, size, size)

    def _ssim(self, x, y, ws=11):
        C1, C2 = 0.01 ** 2, 0.03 ** 2
        k  = self._gaussian_kernel(ws, 1.5, x.device)
        p  = ws // 2
        mu_x   = F.conv2d(x,   k, padding=p)
        mu_y   = F.conv2d(y,   k, padding=p)
        sig_x  = F.conv2d(x*x, k, padding=p) - mu_x * mu_x
        sig_y  = F.conv2d(y*y, k, padding=p) - mu_y * mu_y
        sig_xy = F.conv2d(x*y, k, padding=p) - mu_x * mu_y
        num = (2 * mu_x * mu_y + C1) * (2 * sig_xy + C2)
        den = (mu_x**2 + mu_y**2 + C1) * (sig_x + sig_y + C2)
        return (num / den).mean()

    def forward(self, pred, target):
        mse_loss  = self.mse(pred, target)
        ssim_loss = 1.0 - self._ssim(pred, target)
        return self.alpha * mse_loss + (1 - self.alpha) * ssim_loss


criterion = CombinedLoss(alpha=0.84)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
print("Loss + optimizer + scheduler ready.")


## 6. PSNR & SSIM Metrics

In [ ]:
def compute_psnr(pred, target, max_val=1.0):
    """Peak Signal-to-Noise Ratio (dB)."""
    mse = F.mse_loss(pred, target).item()
    return float("inf") if mse < 1e-12 else 10.0 * math.log10(max_val ** 2 / mse)


def compute_ssim(pred, target, ws=11):
    """Structural Similarity Index (mean over batch)."""
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    coords = torch.arange(ws, dtype=torch.float32, device=pred.device) - ws // 2
    g = torch.exp(-(coords ** 2) / (2 * 1.5 ** 2))
    g /= g.sum()
    k = (g.unsqueeze(0) * g.unsqueeze(1)).unsqueeze(0).unsqueeze(0)
    p = ws // 2
    mu_x   = F.conv2d(pred,   k, padding=p)
    mu_y   = F.conv2d(target, k, padding=p)
    sig_x  = F.conv2d(pred*pred,     k, padding=p) - mu_x * mu_x
    sig_y  = F.conv2d(target*target, k, padding=p) - mu_y * mu_y
    sig_xy = F.conv2d(pred*target,   k, padding=p) - mu_x * mu_y
    ssim   = ((2*mu_x*mu_y + C1) * (2*sig_xy + C2)) / \
             ((mu_x**2 + mu_y**2 + C1) * (sig_x + sig_y + C2))
    return ssim.mean().item()


print("compute_psnr, compute_ssim defined.")


## 7. Training & Validation Loop

In [ ]:
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

def train_one_epoch(model, loader, optimizer, criterion, device, epoch):
    model.train()
    run_loss = run_psnr = 0.0
    N = len(loader)
    for i, (x, y_true) in enumerate(loader):
        x, y_true = x.to(device), y_true.to(device)
        optimizer.zero_grad()
        # Forward pass in float16 via AMP, loss computed in float32
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            y_pred = model(x)
            loss   = criterion(y_pred.float(), y_true.float())
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        run_loss += loss.item()
        run_psnr += compute_psnr(y_pred.detach().float(), y_true)
        if (i + 1) % max(1, N // 5) == 0:
            print(f"  Epoch {epoch} [{i+1}/{N}]  "
                  f"loss={run_loss/(i+1):.4f}  PSNR={run_psnr/(i+1):.2f}dB")
    return {"loss": run_loss / N, "psnr": run_psnr / N}


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    run_loss = run_psnr = run_ssim = 0.0
    N = len(loader)
    for x, y_true in loader:
        x, y_true = x.to(device), y_true.to(device)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            y_pred = model(x)
        y_pred = y_pred.float()   # cast back for metrics
        run_loss += criterion(y_pred, y_true).item()
        run_psnr += compute_psnr(y_pred, y_true)
        run_ssim += compute_ssim(y_pred, y_true)
    return {"loss": run_loss / N, "psnr": run_psnr / N, "ssim": run_ssim / N}


print(f'AMP enabled: {USE_AMP} | GradScaler ready')
print('train_one_epoch, validate defined.')


## 7b. Resume from Checkpoint (Kaggle)

To continue training from a saved checkpoint:
1. Go to your **previous Kaggle notebook** -> Output tab -> click the **three-dot menu** on `last_checkpoint.pth` -> **Add to current notebook as Input**
2. Set `RESUME_CKPT_PATH` in the config cell to the mounted path, e.g.:
   ```python
   RESUME_CKPT_PATH = '/kaggle/input/<your-notebook-slug>/last_checkpoint.pth'
   ```
3. Re-run all cells. The training loop will pick up from the last epoch.

> **Note:** `best_model.pth` is for **inference only** (weights only).  
> `last_checkpoint.pth` is for **resuming training** (full state, saved every epoch).

In [ ]:
start_epoch = 1
best_psnr   = 0.0
history     = {"train_loss": [], "val_loss": [], "val_psnr": [], "val_ssim": []}

if RESUME_CKPT_PATH and os.path.exists(RESUME_CKPT_PATH):
    print(f'Loading checkpoint: {RESUME_CKPT_PATH}')
    ckpt = torch.load(RESUME_CKPT_PATH, map_location=DEVICE)

    # Load model weights
    if isinstance(ckpt, dict) and 'model' in ckpt:
        # Full checkpoint (saved with save_checkpoint utility)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        start_epoch = ckpt.get('epoch', 1) + 1
        best_psnr   = ckpt.get('best_psnr', 0.0)
        history     = ckpt.get('history', history)
    else:
        # state_dict only (saved with torch.save(model.state_dict(), ...))
        model.load_state_dict(ckpt)
        print('  Loaded weights only (no optimizer state / epoch info)')

    # Re-align scheduler to the resumed epoch
    for _ in range(start_epoch - 1):
        scheduler.step()

    print(f'Resumed from epoch {start_epoch} | best PSNR so far: {best_psnr:.2f} dB')
else:
    print('No checkpoint found - training from scratch.')


In [ ]:
BEST_MODEL_PATH = os.path.join(CKPT_DIR, 'best_model.pth')      # weights only - for inference
LAST_CKPT_PATH  = os.path.join(CKPT_DIR, 'last_checkpoint.pth')  # full state   - for resuming

for epoch in range(start_epoch, start_epoch + EPOCHS):
    print(f"\n{'='*55}")
    print(f" Epoch {epoch}  (run epochs {start_epoch} - {start_epoch+EPOCHS-1})")
    print(f"{'='*55}")
    tr = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, epoch)
    va = validate(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history["train_loss"].append(tr["loss"])
    history["val_loss"].append(va["loss"])
    history["val_psnr"].append(va["psnr"])
    history["val_ssim"].append(va["ssim"])

    print(f"  Train loss={tr['loss']:.4f}  PSNR={tr['psnr']:.2f}dB")
    print(f"  Val   loss={va['loss']:.4f}  PSNR={va['psnr']:.2f}dB  SSIM={va['ssim']:.4f}")

    # ── Best model: weights only, saved when PSNR improves (for inference) ────────
    if va["psnr"] > best_psnr:
        best_psnr = va["psnr"]
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  [BEST] PSNR={best_psnr:.2f}dB -> {BEST_MODEL_PATH}")

    # ── Last checkpoint: full state, saved every epoch (for resuming) ──────────
    torch.save({
        'epoch':      epoch,
        'best_psnr':  best_psnr,
        'model':      model.state_dict(),
        'optimizer':  optimizer.state_dict(),
        'history':    history,
    }, LAST_CKPT_PATH)

print(f"\nRun complete. Epochs {start_epoch} - {start_epoch+EPOCHS-1} done.")
print(f"Best Val PSNR : {best_psnr:.2f} dB")
print(f"Inference     : {BEST_MODEL_PATH}  (weights only)")
print(f"Resume        : {LAST_CKPT_PATH}  (full state, last epoch)")


## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ep = range(1, EPOCHS + 1)

axes[0].plot(ep, history["train_loss"], label="Train", marker="o")
axes[0].plot(ep, history["val_loss"],   label="Val",   marker="s")
axes[0].set_title("Combined Loss (MSE + SSIM)"); axes[0].set_xlabel("Epoch")
axes[0].legend(); axes[0].grid(True)

axes[1].plot(ep, history["val_psnr"], color="green", marker="o")
axes[1].set_title("Val PSNR (dB)"); axes[1].set_xlabel("Epoch")
axes[1].grid(True)

axes[2].plot(ep, history["val_ssim"], color="purple", marker="o")
axes[2].set_title("Val SSIM"); axes[2].set_xlabel("Epoch")
axes[2].grid(True)

plt.suptitle("ConvNeXt-Temporal Transformer - Training Curves", fontsize=13)
plt.tight_layout()

curves_path = os.path.join(CKPT_DIR, 'training_curves.png')
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
print(f'Curves saved -> {curves_path}')
plt.show()


## 9. Visual Inference - Single Sample

In [ ]:
def load_single_sample(sid):
    """Load one sample by ID, return (x, y) with batch dimension."""
    ds = Inter4KDataset(DATASET_ROOT, [sid])
    x, y = ds[0]
    return x.unsqueeze(0), y.unsqueeze(0)


# Load best model (weights only)
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
model.eval()

DEMO_ID = 1   # change to any valid sample ID
x_d, y_d = load_single_sample(DEMO_ID)

with torch.no_grad():
    y_p = model(x_d.to(DEVICE)).cpu()

psnr_v = compute_psnr(y_p, y_d)
ssim_v = compute_ssim(y_p, y_d)

fig, axes = plt.subplots(1, T + 2, figsize=(3 * (T + 2), 3.5))
for t in range(T):
    axes[t].imshow(x_d[0, t, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[t].set_title(f"Input t={t}", fontsize=8)
    axes[t].axis("off")

axes[T].imshow(y_d[0, 0].numpy(), cmap="gray", vmin=0, vmax=1)
axes[T].set_title("Target (GT)", fontsize=8)
axes[T].axis("off")

axes[T + 1].imshow(y_p[0, 0].numpy(), cmap="gray", vmin=0, vmax=1)
axes[T + 1].set_title(f"Predicted\nPSNR={psnr_v:.2f}dB\nSSIM={ssim_v:.4f}", fontsize=8)
axes[T + 1].axis("off")

plt.suptitle(f"Sample {DEMO_ID} - ConvNeXt-Temporal Transformer Output", fontsize=11)
plt.tight_layout()
plt.show()


## 10. Checkpoint Utilities

In [ ]:
def save_checkpoint(model, optimizer, epoch, path):
    torch.save({"epoch": epoch, "model": model.state_dict(),
                "optimizer": optimizer.state_dict()}, path)
    print(f"Saved -> {path}")


def load_checkpoint(model, optimizer, path, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    print(f"Loaded from {path}  (epoch {ckpt['epoch']})")
    return ckpt["epoch"]


print("save_checkpoint, load_checkpoint defined.")
